# Transfer Learning with MobileNetV2 - نسخة المحاضر

هذا الدفتر مخصص للمحاضر، مع شرح تفصيلي لكل خطوة: ماذا نفعل، ولماذا، وكيف نفسر النتائج للطلاب.

## الهدف التعليمي
استخدام **MobileNetV2** المدرب مسبقاً على ImageNet لتصنيف MNIST.
نُجمّد الطبقات المدربة وندرب **رأس تصنيف** جديد فقط.

## خطة الشرح
1. استيراد المكتبات
2. قراءة البيانات
3. تجهيز الصور (224×224 RGB)
4. تقسيم البيانات
5. بناء MobileNetV2
6. تجميد Base
7. تدريب الرأس
8. التقييم
9. عرض تنبؤات


## الخطوة 1: استيراد المكتبات

MobileNetV2 من `applications` — نموذج خفيف مدرب على ImageNet.


In [ ]:
# الخطوة 1) استيراد المكتبات
# pip install tensorflow -q
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input


## الخطوة 2: قراءة البيانات


In [ ]:
# Step 2) قراءة البيانات / Load dataset
import os
import urllib.request

filename = 'mnist_sample.csv'
if not os.path.exists(filename):
    url = 'https://raw.githubusercontent.com/iksasa15/AI-ML/main/code/15-%20Deep%20Learning/6-%20Transfer%20Learning/mnist_sample.csv'
    urllib.request.urlretrieve(url, filename)

dataset = pd.read_csv(filename)
dataset.head()


## الخطوة 3: تجهيز الصور لـ MobileNetV2

MobileNetV2 يتوقع:
- حجم **224×224**
- **3 قنوات RGB** (MNIST رمادي → نكرر 3 مرات)
- **preprocess_input**: توافق مع weights='imagenet'


In [ ]:
# الخطوة 3) تجهيز الصور
y = dataset['label'].values.astype(int)
pixel_cols = [c for c in dataset.columns if c.startswith('pixel_')]
X_gray = dataset[pixel_cols].values.reshape(-1, 28, 28, 1).astype('float32')

# Grayscale -> RGB and resize to 224x224
X_rgb = np.repeat(X_gray, 3, axis=-1)
X_resized = tf.image.resize(X_rgb, (224, 224)).numpy()
X_resized = preprocess_input(X_resized)
print('Prepared shape:', X_resized.shape)


## الخطوة 4: تقسيم البيانات

`stratify=y` للحفاظ على توازن الأرقام 0–9.


In [ ]:
# الخطوة 4) تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X_resized, y, test_size=0.2, random_state=0, stratify=y
)


## الخطوة 5: بناء النموذج

- **base_model**: MobileNetV2 بدون رأس (include_top=False)
- **GlobalAveragePooling2D**: يختصر feature maps لمتجه
- **Dense(10, softmax)**: 10 أرقام


In [ ]:
# الخطوة 5) بناء MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
outputs = Dense(10, activation='softmax')(x)
model = Model(inputs, outputs)
model.summary()


## الخطوة 6: تجميد Base

`base_model.trainable = False` — لا نحدّث أوزان MobileNetV2.
ندرب فقط GlobalAveragePooling + Dense (رأس جديد).
**فائدة:** أسرع وأقل بيانات مطلوبة.


In [ ]:
# الخطوة 6) تجميد Base
base_model.trainable = False
print(f'Trainable layers: {sum([l.trainable for l in model.layers])}')


## الخطوة 7: تدريب الرأس

5 epochs كافية لأننا ندرب طبقات قليلة فقط.


In [ ]:
# الخطوة 7) تدريب الرأس
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=16,
    verbose=1
)


## الخطوة 8: التقييم


In [ ]:
# الخطوة 8) التقييم
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss: {loss:.4f}')
print(f'Test accuracy: {accuracy:.2%}')


## الخطوة 9: عرض التنبؤات

**ملاحظة للمحاضر:** MNIST ≠ صور ImageNet — Transfer Learning هنا تعليمي.
في التطبيق الحقيقي نستخدم domain مشابه (صور طبيعية).


In [ ]:
# الخطوة 9) عرض تنبؤات
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_test[i].astype('float32') * 0.5 + 0.5)
    color = 'green' if y_pred[i] == y_test[i] else 'red'
    ax.set_title(f'True: {y_test[i]} | Pred: {y_pred[i]}', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Transfer Learning Predictions (MobileNetV2)')
plt.tight_layout()
plt.show()
